
# Comparativa de Modelos de Machine Learning sobre Datasets UCI

Este notebook implementa un flujo experimental completo y reproducible para comparar múltiples algoritmos de Machine Learning sobre cinco datasets del repositorio UCI Machine Learning Repository.

## Datasets utilizados

1. Human Activity Recognition Using Smartphones
2. Heart Disease
3. Default of Credit Card Clients
4. Bike Sharing
5. Computer Hardware

## Modelos evaluados

- Random Forest
- Support Vector Machine (SVM)
- K-Nearest Neighbors (KNN)
- Multi-Layer Perceptron (MLP)
- XGBoost
- LightGBM
- Naive Bayes
- Decision Tree
- Logistic Regression

## Métricas evaluadas

- Accuracy
- Balanced Accuracy
- Precision
- Recall
- F1-Score
- Matriz de confusión


In [ ]:

# Instalación de dependencias (ejecutar si es necesario)
# !pip install pandas numpy scikit-learn matplotlib seaborn xgboost lightgbm ucimlrepo openpyxl


In [ ]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import matplotlib.pyplot as plt
import seaborn as sns

import os


In [ ]:

# Configuración de datasets UCI

datasets = {
    "HAR": 240,
    "HeartDisease": 45,
    "CreditDefault": 350,
    "BikeSharing": 275,
    "ComputerHardware": 29
}

print("Datasets configurados:")
datasets


In [ ]:

def preprocess_data(X, y):

    X = X.copy()

    categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

    # Transformar variables categóricas
    for col in categorical_cols:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))

    # Imputación de valores faltantes
    imputer = SimpleImputer(strategy='mean')
    X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

    # Escalamiento
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Procesamiento del target
    if y.dtype == 'object':
        y = LabelEncoder().fit_transform(y.astype(str))

    return X_scaled, y


In [ ]:

models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),

    "SVM": SVC(
        kernel='rbf',
        probability=True,
        random_state=42
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "MLP": MLPClassifier(
        hidden_layer_sizes=(100, 50),
        max_iter=500,
        random_state=42
    ),

    "NaiveBayes": GaussianNB(),

    "DecisionTree": DecisionTreeClassifier(
        random_state=42
    ),

    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        eval_metric='logloss',
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        random_state=42
    )
}

list(models.keys())


In [ ]:

results = []
confusion_matrices = {}

for dataset_name, dataset_id in datasets.items():

    print("=" * 80)
    print(f"Procesando dataset: {dataset_name}")
    print("=" * 80)

    try:
        dataset = fetch_ucirepo(id=dataset_id)

        X = dataset.data.features
        y = dataset.data.targets

        if isinstance(y, pd.DataFrame):
            y = y.iloc[:, 0]

        X_processed, y_processed = preprocess_data(X, y)

        X_train, X_test, y_train, y_test = train_test_split(
            X_processed,
            y_processed,
            test_size=0.2,
            random_state=42,
            stratify=y_processed if len(np.unique(y_processed)) < 20 else None
        )

        for model_name, model in models.items():

            print(f"Entrenando modelo: {model_name}")

            try:
                model.fit(X_train, y_train)

                y_pred = model.predict(X_test)

                accuracy = accuracy_score(y_test, y_pred)
                balanced_acc = balanced_accuracy_score(y_test, y_pred)

                precision = precision_score(
                    y_test,
                    y_pred,
                    average='weighted',
                    zero_division=0
                )

                recall = recall_score(
                    y_test,
                    y_pred,
                    average='weighted',
                    zero_division=0
                )

                f1 = f1_score(
                    y_test,
                    y_pred,
                    average='weighted',
                    zero_division=0
                )

                cm = confusion_matrix(y_test, y_pred)

                results.append({
                    "Dataset": dataset_name,
                    "Model": model_name,
                    "Accuracy": accuracy,
                    "BalancedAccuracy": balanced_acc,
                    "Precision": precision,
                    "Recall": recall,
                    "F1Score": f1
                })

                confusion_matrices[(dataset_name, model_name)] = cm

            except Exception as model_error:
                print(f"Error en {model_name}: {model_error}")

    except Exception as dataset_error:
        print(f"Error en dataset {dataset_name}: {dataset_error}")


In [ ]:

results_df = pd.DataFrame(results)

print(results_df.head())

results_df.to_csv("resultados_experimentos.csv", index=False)

print("\nResultados exportados correctamente.")


In [ ]:

# Visualización comparativa

plt.figure(figsize=(16, 8))

sns.barplot(
    data=results_df,
    x='Dataset',
    y='F1Score',
    hue='Model'
)

plt.title("Comparación de F1-Score por Dataset y Modelo")
plt.xticks(rotation=15)
plt.tight_layout()

plt.show()


In [ ]:

# Mostrar matrices de confusión

for key, cm in confusion_matrices.items():

    dataset_name, model_name = key

    plt.figure(figsize=(6, 5))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues'
    )

    plt.title(f"Matriz de Confusión\n{dataset_name} - {model_name}")
    plt.xlabel("Predicción")
    plt.ylabel("Valor Real")

    plt.tight_layout()
    plt.show()



# Conclusiones

Este flujo experimental permite:

- Comparar múltiples algoritmos de Machine Learning.
- Evaluar desempeño sobre distintos tipos de datasets.
- Analizar estabilidad y robustez de modelos.
- Exportar resultados reproducibles.
- Visualizar matrices de confusión y métricas comparativas.

El flujo puede extenderse fácilmente incorporando:
- Validación cruzada.
- Optimización de hiperparámetros.
- Feature engineering.
- Interpretabilidad de modelos.
- Métricas adicionales.
